# ART SLIDE — アイコンデザインスタジオ
原画から30案を生成し、一覧から選んで手動編集できます。ファイル保存は行いません。


In [ ]:
from PIL import Image, ImageOps, ImageDraw, ImageFilter
import io, requests, random, math, json
from IPython.display import display
from google.colab import files

# 原画は Wikimedia Commons のパブリックドメイン作品。取得失敗時は手元の画像をアップロード。
URL = 'https://upload.wikimedia.org/wikipedia/commons/6/6a/Mona_Lisa.jpg'
try:
    r = requests.get(URL, headers={'User-Agent':'ArtSlideNotebook/1.0 (educational image puzzle)'}, timeout=30)
    r.raise_for_status()
    original = Image.open(io.BytesIO(r.content)).convert('RGB')
except Exception as e:
    print('原画をアップロードしてください:', e)
    uploaded = files.upload()
    original = Image.open(io.BytesIO(next(iter(uploaded.values())))).convert('RGB')
display(original.copy().resize((320, round(320*original.height/original.width))))

In [ ]:
# 原画に対する相対座標 (左, 上, 右, 下)。正方形に切り出す。
# 必要なら数値を調整。右目・口の元ピース番号は次のプレビューで確認。
SIZE = 1200
N = 5
T = SIZE // N
w,h = original.size
side = min(w*0.77,h*0.58)
cx,cy = w*0.50,h*0.29
x = max(0,min(w-side,cx-side/2))
y = max(0,min(h-side,cy-side/2))
art = original.crop((round(x),round(y),round(x+side),round(y+side))).resize((SIZE,SIZE),Image.Resampling.LANCZOS)
tiles = [art.crop((c*T,r*T,(c+1)*T,(r+1)*T)) for r in range(N) for c in range(N)]
preview = art.copy()
d = ImageDraw.Draw(preview)
for i in range(25):
    r,c=divmod(i,N)
    d.rectangle((c*T,r*T,(c+1)*T-1,(r+1)*T-1),outline='white',width=3)
    d.rectangle((c*T+5,r*T+5,c*T+44,r*T+30),fill='black')
    d.text((c*T+10,r*T+9),str(i+1),fill='white')
display(preview.resize((420,420)))

In [ ]:
# ピース番号は上のプレビューで確認してください（1始まり）
RIGHT_EYE = 14 # @param {type:"integer"}
MOUTH = 18 # @param {type:"integer"}
SEED = 42 # @param {type:"integer"}
TARGETS = {RIGHT_EYE-1: 13, MOUTH-1: 17}
BLANK = 24
assert 1<=RIGHT_EYE<=24 and 1<=MOUTH<=24 and len(TARGETS)==2
def neighbors(p):
    r,c=divmod(p,5)
    return [rr*5+cc for rr,cc in [(r-1,c),(r+1,c),(r,c-1),(r,c+1)] if 0<=rr<5 and 0<=cc<5]

def solvable(b):
    seq=[x for x in b if x!=BLANK]
    return sorted(b)==list(range(25)) and sum(a>z for i,a in enumerate(seq) for z in seq[i+1:])%2==0

def generate(seed=42,count=30):
    rng=random.Random(seed)
    brightness=[t.convert('L').resize((1,1)).getpixel((0,0)) for t in tiles]
    fixed=set(TARGETS.values())
    movable=[i for i in range(25) if i not in fixed]
    remaining=[i for i in range(25) if i not in TARGETS and i!=BLANK]
    found=[]; seen=set()
    for attempt in range(20000):
        blank=rng.choice([i for i in [4,9,14,19,21,22,23,24,0,1,2,3] if i not in fixed])
        b=[None]*25
        for tile,pos in TARGETS.items(): b[pos]=tile
        b[blank]=BLANK
        pool=remaining.copy(); rng.shuffle(pool)
        free=[i for i in movable if i!=blank]
        for i,t in zip(free,pool): b[i]=t
        # 奇数幅の盤面は反転数が偶数なら必ず解ける。
        if not solvable(b):
            a,z=free[:2]; b[a],b[z]=b[z],b[a]
        if blank>=5 and brightness[b[blank-5]]<95 and attempt<10000: continue
        if tuple(b) in seen: continue
        if any(sum(a!=z for a,z in zip(b,old))<10 for old in found): continue
        assert solvable(b)
        found.append(b); seen.add(tuple(b))
        if len(found)==count: return found
    raise RuntimeError('候補不足: ピース番号を確認してください')
variants=generate(SEED)
assert len(variants)==30 and all(solvable(b) for b in variants)
print('30案生成済み・全案解けることを検証済み')


In [ ]:
# ピースの絵柄は変更せず、縁の陰影だけを追加する。
from PIL import ImageEnhance
def render(board, gap=4, radius=10, outer_radius=240):
    out=Image.new('RGBA',(SIZE,SIZE),(59,39,25,255))
    for p,tile in enumerate(board):
        if tile==BLANK: continue
        r,c=divmod(p,5)
        piece=tiles[tile].resize((T-gap,T-gap),Image.Resampling.LANCZOS).convert('RGBA')
        mask=Image.new('L',piece.size)
        ImageDraw.Draw(mask).rounded_rectangle((0,0,piece.width-1,piece.height-1),radius,fill=255)
        overlay=Image.new('RGBA',piece.size)
        d=ImageDraw.Draw(overlay)
        for k in range(5):
            d.rounded_rectangle((k,k,piece.width-1-k,piece.height-1-k),max(1,radius-k),outline=(255,220,170, max(0,65-k*12)) if k<2 else (30,14,5, max(0,45-k*7)),width=1)
        piece=Image.alpha_composite(piece,overlay)
        out.paste(piece,(c*T+gap//2,r*T+gap//2),mask)
    mask=Image.new('L',out.size)
    ImageDraw.Draw(mask).rounded_rectangle((0,0,SIZE-1,SIZE-1),outer_radius,fill=255)
    out.putalpha(mask)
    return out

# 30案の一覧と手動編集。Colabの出力内で操作できます。
import base64, html
from IPython.display import HTML, display
def data_url(im):
    f=io.BytesIO(); im.save(f,format='PNG')
    return 'data:image/png;base64,'+base64.b64encode(f.getvalue()).decode()
# 各ピースは元画像の一部分をそのまま使用。
tile_urls=[data_url(t.resize((240,240))) for t in tiles]
thumbs=[data_url(render(b).resize((220,220),Image.Resampling.LANCZOS)) for b in variants]
solved_url=data_url(render(list(range(25))).resize((220,220)))
payload=json.dumps(dict(boards=variants,tiles=tile_urls,thumbs=thumbs,solved=solved_url,targets=TARGETS))
page=r"""
<div class="as-root">
<style>
.as-root{font:14px system-ui,sans-serif;color:#28231e;background:#f7f4ef;padding:16px;border-radius:12px}
.as-root *{box-sizing:border-box}
.as-root button,.as-root select{font:inherit;cursor:pointer;padding:8px 12px;border:1px solid #c8bfb2;border-radius:7px;background:white;color:#28231e}
.as-root button:hover{background:#eee8df}
.as-root .studio{display:grid;grid-template-columns:minmax(260px,440px) minmax(240px,1fr);gap:22px;align-items:start}
.as-root .gallery{display:grid;grid-template-columns:repeat(auto-fit,minmax(105px,1fr));gap:10px;max-height:650px;overflow:auto;padding:3px}
.as-root .candidate{padding:4px;display:grid;gap:4px;text-align:center}
.as-root .candidate.active{outline:2px solid #996b36;background:#eadbc7}
.as-root .candidate img{width:100%;display:block;border-radius:18%}
.as-root .board{display:grid;grid-template-columns:repeat(5,1fr);gap:2px;background:#3b2719;border-radius:19%;overflow:hidden;aspect-ratio:1;max-width:440px}
.as-root .piece{padding:0;border:0;border-radius:4px;position:relative;aspect-ratio:1;background-size:cover;box-shadow:inset 1px 1px 3px #ffffff44,inset -2px -2px 4px #0005}
.as-root .piece.empty{background:#493023;box-shadow:inset 2px 3px 8px #0009}
.as-root .piece.selected{outline:3px solid #e7b34f;outline-offset:-4px}
.as-root .tools{display:flex;gap:6px;flex-wrap:wrap;margin:10px 0}
.as-root .status{padding:8px 0;font-weight:600}
.as-root .reference{display:flex;gap:12px;align-items:center;margin-top:10px}
.as-root .reference img{width:90px}
@media(max-width:650px){.as-root .studio{grid-template-columns:1fr}.as-root .gallery{max-height:420px}}
</style>
<div class="studio">
<section><h3 style="margin-top:0">手動編集</h3>
<div class="board"></div>
<div class="tools"><select class="mode"><option value="slide">スライド（合法手）</option><option value="swap">自由入れ替え（2枚選択）</option></select><button class="undo">元に戻す</button><button class="reset">リセット</button><button class="solve">完成状態</button></div>
<label><input type="checkbox" class="numbers"> ピース番号を表示</label>
<div class="status"></div><div class="hint"></div>
<div class="reference"><img class="solved"><span>完成状態の見本<br>右目と口は自動生成時のみ固定。手動編集では自由に変更できます。</span></div>
</section>
<section><h3 style="margin-top:0">生成候補 30案</h3><div style="margin-bottom:8px">クリックすると左の編集画面に反映されます。</div><div class="gallery"></div></section>
</div>
<script>
(()=>{
const root=document.currentScript.parentElement;
const data=__PAYLOAD__;
const q=s=>root.querySelector(s);
let idx=0,b=data.boards[0].slice(),history=[],selected=-1;
const solved=()=>Array.from({length:25},(_,i)=>i);
const isSolvable=()=>{const a=b.filter(x=>x!==24);let n=0;for(let i=0;i<a.length;i++)for(let j=i+1;j<a.length;j++)if(a[i]>a[j])n++;return n%2===0};
const adjacent=(a,z)=>Math.abs(Math.floor(a/5)-Math.floor(z/5))+Math.abs(a%5-z%5)===1;
const exchange=(a,z)=>{history.push(b.slice());[b[a],b[z]]=[b[z],b[a]];selected=-1;draw()};
function draw(){
 q('.board').replaceChildren();
 b.forEach((t,i)=>{
  const el=document.createElement('button');el.className='piece'+(t===24?' empty':'')+(i===selected?' selected':'');
  if(t!==24)el.style.backgroundImage='url('+data.tiles[t]+')';
  el.title=t===24?'空白':'ピース '+(t+1);
  if(q('.numbers').checked){el.textContent=t===24?'空':t+1;el.style.color='white';el.style.textShadow='0 1px 3px black,0 0 4px black'}
  el.onclick=()=>{if(q('.mode').value==='slide'){const z=b.indexOf(24);if(adjacent(i,z))exchange(i,z)}else{if(selected===i){selected=-1;draw()}else if(selected<0){selected=i;draw()}else exchange(selected,i)}};
  q('.board').append(el);
 });
 const ok=isSolvable();q('.status').textContent=(idx>=0?'案 '+(idx+1)+'　':'完成見本　')+(ok?'解ける配置':'解けない配置（奇数回の入れ替え）');q('.status').style.color=ok?'#37634b':'#a64032';
 q('.hint').textContent=q('.mode').value==='slide'?'空白の隣をクリックして移動します。':'任意の2マスをクリックして交換します。解けない場合はもう1組を交換してください。';
 q('.undo').disabled=!history.length;
 root.querySelectorAll('.candidate').forEach((el,i)=>el.classList.toggle('active',i===idx));
}
function load(i){idx=i;b=data.boards[i].slice();history=[];selected=-1;draw()}
data.thumbs.forEach((url,i)=>{const el=document.createElement('button');el.className='candidate';const im=document.createElement('img');im.src=url;im.alt='案 '+(i+1);el.append(im,document.createTextNode(String(i+1).padStart(2,'0')));el.onclick=()=>load(i);q('.gallery').append(el)});
q('.solved').src=data.solved;
q('.mode').onchange=()=>{selected=-1;draw()};
q('.numbers').onchange=draw;
q('.undo').onclick=()=>{if(history.length){b=history.pop();selected=-1;draw()}};
q('.reset').onclick=()=>{if(idx>=0)load(idx);else{b=solved();history=[];selected=-1;draw()}};
q('.solve').onclick=()=>{history.push(b.slice());b=solved();selected=-1;draw()};
draw();
})();
</script></div>
"""
display(HTML(page.replace('__PAYLOAD__',payload)))
